# 05 - Baseline Model Evaluation

This notebook evaluates all baseline models on both experimental tasks:

- HDFS log anomaly classification;
- QA response quality evaluation.

Evaluated models include:

- Base model;
- CPT;
- SFT Logs LoRA;
- SFT QA LoRA;
- CPT → SFT Logs;
- CPT → SFT QA;
- Multi-task LoRA.

Results are saved to `artifacts/results/baselines/`.

In [ ]:
import exp_common as E
from pathlib import Path
import json, pandas as pd

RUN_SEEDS = [42, 123, 777] 
EVAL_LOGS_N = 2000
EVAL_QA_N = 500
BATCH_LOGS = 16
BATCH_QA = 4

In [ ]:
logs_eval = E.read_jsonl(E.PREPARED_DIR/'logs_test_eval.jsonl')[:EVAL_LOGS_N]
qa_eval = E.read_jsonl(E.PREPARED_DIR/'qa_test_eval.jsonl')[:EVAL_QA_N]
out_dir = E.RESULTS_DIR / 'baselines'
out_dir.mkdir(parents=True, exist_ok=True)

def eval_one(name, adapter_dir, seed='base'):
    print('Evaluating', name, seed, adapter_dir)
    logs_metrics = E.evaluate_logs(adapter_dir, logs_eval, out_dir/f'{name}_seed_{seed}_logs.json', batch_size=BATCH_LOGS)
    qa_metrics = E.evaluate_qa(adapter_dir, qa_eval, out_dir/f'{name}_seed_{seed}_qa.json', batch_size=BATCH_QA)
    return {'model': name, 'seed': seed, **{f'logs_{k}': v for k, v in logs_metrics.items() if not isinstance(v, list)}, **{f'qa_{k}': v for k, v in qa_metrics.items()}}

rows = []
# Base has no adapter. Evaluate once.
rows.append(eval_one('base', None, seed='none'))

for seed in RUN_SEEDS:
    candidates = {
        'cpt': E.ADAPTERS_DIR/'cpt'/f'seed_{seed}'/'adapter',
        'ft_logs': E.ADAPTERS_DIR/'logs'/f'seed_{seed}'/'adapter',
        'ft_qa': E.ADAPTERS_DIR/'qa'/f'seed_{seed}'/'adapter',
        'cpt_ft_logs': E.ADAPTERS_DIR/'logs_on_cpt'/f'seed_{seed}'/'adapter',
        'cpt_ft_qa': E.ADAPTERS_DIR/'qa_on_cpt'/f'seed_{seed}'/'adapter',
        'multitask_lora': E.ADAPTERS_DIR/'multitask'/f'seed_{seed}'/'adapter',
        'multitask_lora_on_cpt': E.ADAPTERS_DIR/'multitask_on_cpt'/f'seed_{seed}'/'adapter',
    }
    for name, ad in candidates.items():
        if (ad/'adapter_model.safetensors').exists():
            rows.append(eval_one(name, ad, seed=seed))
        else:
            print('Missing, skip:', ad)

df = pd.DataFrame(rows)
if (out_dir/'baseline_metrics_raw.csv').exists():
    existing = pd.read_csv(out_dir/'baseline_metrics_raw.csv')
    df = pd.concat([existing, df], ignore_index=True).drop_duplicates(subset=['model', 'seed'])
df.to_csv(out_dir/'baseline_metrics_raw.csv', index=False)
df

In [ ]:
# mean ± std
num_cols = [c for c in df.columns if c not in ['model', 'seed'] and pd.api.types.is_numeric_dtype(df[c])]
summary = df.groupby('model')[num_cols].agg(['mean', 'std'])
summary.to_csv(out_dir/'baseline_metrics_mean_std.csv')
summary